# TEST — Hourly (clte) on polytope-test.mn5

Tests the new test server `polytope-test.mn5.apps.dte.destination-earth.eu`
with the hourly portfolio using `IFS-NEMO` and `2t`.

**Prerequisites:** run `01_key_destine_once.ipynb` once to authenticate.

In [ ]:
import logging, warnings
import earthkit.data

earthkit.data.config.set("cache-policy", "off")

for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
from polytope_zarr import PolytopeZarrStore

## 1. Create hourly store on test server

In [ ]:
TEST_ADDRESS = "polytope-test.mn5.apps.dte.destination-earth.eu"

store = PolytopeZarrStore.from_climate_dt(
    models=["IFS-NEMO"],
    experiment="hist",
    levtype="sfc",
    frequency="hourly",
    resolution="high",
    start_date="1990-02-01T00:00:00",
    end_date="1990-02-02T23:00:00",
    address=TEST_ADDRESS,
)
print(store)

In [ ]:
ds = store.open()
store._batch_size=1 # you can enforce not to get full days
ds

## 2. Fetch a single hourly field (lazy browse)

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt

field = ds["2t"].sel(model="IFS-NEMO", time="1990-02-01T12:00")
print(f"Fetching {dict(field.sizes)} values...")

hp.mollview(field.values, title="IFS-NEMO — 2m temperature — 1990-02-01 12:00 (test server)",
            unit="K", cmap="RdYlBu_r", nest=True, flip='geo')
plt.show()

## 3. Server-side area subsetting

In [ ]:
store._batch_size=23 # enforce one polytope request for this cell
area_hourly = ds["2t"].polytope.sel(
    model="IFS-NEMO",
    time=slice("1990-02-01T00:00", "1990-02-01T23:00"),
    area=(55, 5, 47, 15),
)
area_hourly

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

var_name = list(area_hourly.data_vars)[0]
field_noon = area_hourly[var_name].sel(time="1990-02-01T12:00")

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()}, figsize=(8, 6))
field_noon.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="RdYlBu_r",
                cbar_kwargs={"label": "K"})
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
ax.set_title(f"IFS-NEMO — {var_name} — 1990-02-01 12:00 (test server)\n(Central Europe, 0.25° grid)")
plt.tight_layout()
plt.show()

## 4. Feature extraction — timeseries (hourly)

Hourly timeseries uses `time_axis: "date"` and returns GRIB.
This tests whether `polytope-test.mn5` has fixed the gribjump bug.

In [ ]:
# Point timeseries — hourly (clte stream)
ts_result = ds["2t"].polytope.sel(
    model="IFS-NEMO",
    time=slice("1990-02-01T00:00", "1990-02-02T23:00"),
    point=(52.5, 13.4),  # Berlin
)
print(type(ts_result))
ts_result

In [ ]:
import pandas as pd

# Extract times and values from CoverageJSON
cov = ts_result["coverages"][0]
times = pd.to_datetime(cov["domain"]["axes"]["t"]["values"])
values = cov["ranges"]["2t"]["values"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, values, marker="o", markersize=3)
ax.set_ylabel("2t (K)")
ax.set_title("IFS-NEMO — 2m temperature — Berlin (test server, hourly)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Feature extraction — bounding box

In [ ]:
TEST_ADDRESS = "polytope-test.mn5.apps.dte.destination-earth.eu"

store = PolytopeZarrStore.from_climate_dt(
    models=["IFS-NEMO"],
    experiment="hist",
    levtype="sfc",
    resolution="high",
    frequency="hourly",
    start_date="1990-02-01T00:00:00",
    end_date="1990-02-02T23:00:00",
    address=TEST_ADDRESS,
)
print(store)

In [ ]:
ds = store.open()
ds

In [ ]:
# Bounding box — single hour
bbox_result = ds["2t"].polytope.sel(
    model="IFS-NEMO",
    time="1990-02-01T12:00",
    bbox=(47, 5, 55, 15),  # (south, west, north, east)
)
print(type(bbox_result))
bbox_result

In [ ]:
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.collections import PolyCollection
import matplotlib.colors as mcolors
import healpy as hp

# ── Plotting a partial (bbox) HEALPix field ────────────────────────
# HEALPix is globally defined, but we do NOT need the full sky array.
# Each pixel's geometry is fully determined by just (nside, pixel_index),
# so we can compute the exact polygon boundary for any subset of pixels
# without padding, interpolation, or reconstructing the global map.
#
# Steps:
#   1. Convert the returned lat/lon → HEALPix pixel indices (ang2pix)
#   2. Get each pixel's boundary vertices (hp.boundaries) — works per-pixel
#   3. Render as filled polygons (PolyCollection) on a cartopy map

nside = store.nside
da = bbox_result["2t"].squeeze()
lats = bbox_result.coords["latitude"].values
lons = bbox_result.coords["longitude"].values
values = da.values.ravel()

# HEALPix pixel indices (NESTED)
theta = np.radians(90.0 - lats)
phi = np.radians(lons)
pix_ids = hp.ang2pix(nside, theta, phi, nest=True)

# Get pixel boundary vertices — only for *our* pixels, not the full sky
n_steps = 50
boundaries = hp.boundaries(nside, pix_ids, step=n_steps, nest=True)

# Convert boundary xyz → lon/lat and build polygon list
polygons = []
for i in range(len(pix_ids)):
    bx, by, bz = boundaries[i]
    b_lon = np.degrees(np.arctan2(by, bx))
    b_lat = np.degrees(np.arcsin(bz))
    polygons.append(list(zip(b_lon, b_lat)))

# Plot with cartopy — native HEALPix cells, no interpolation
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()}, figsize=(10, 7))

vmin, vmax = np.nanmin(values), np.nanmax(values)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.RdYlBu_r

coll = PolyCollection(polygons, transform=ccrs.PlateCarree(),
                       edgecolors="face", linewidths=0.1)
coll.set_array(values)
coll.set_cmap(cmap)
coll.set_norm(norm)
ax.add_collection(coll)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle="--")
ax.set_extent([4, 16, 46, 56])
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

plt.colorbar(coll, ax=ax, label="K", shrink=0.8, pad=0.02)
ax.set_title("bbox — 2t — 1990-02-01 12:00\n(native HEALPix pixels, no interpolation)")
plt.tight_layout()
plt.show()

## 6. Feature extraction — polygon (country cut-out)

Uses `earthkit.geo.cartography.country_polygons()` to get the boundary
of any named country, then passes it to the Polytope polygon feature.
Returns native HEALPix cells inside the country shape — no interpolation.

In [ ]:
import earthkit.geo.cartography

# Get country boundary from Natural Earth via earthkit-geo
COUNTRY = "Germany"
shapes = earthkit.geo.cartography.country_polygons([COUNTRY], resolution=50e6)

poly_result = ds["2t"].polytope.sel(
    model="IFS-NEMO",
    time="1990-02-01T12:00",
    polygon=shapes,
)
print(f"{COUNTRY}: {len(shapes)} sub-polygon(s), {sum(len(s) for s in shapes)} vertices total")
print(type(poly_result))
poly_result

In [ ]:
nside = store.nside
da = poly_result["2t"].squeeze()
lats = poly_result.coords["latitude"].values
lons = poly_result.coords["longitude"].values
values = da.values.ravel()

# HEALPix pixel indices (NESTED)
theta = np.radians(90.0 - lats)
phi = np.radians(lons)
pix_ids = hp.ang2pix(nside, theta, phi, nest=True)

n_steps = 50
boundaries = hp.boundaries(nside, pix_ids, step=n_steps, nest=True)

polygons = []
for i in range(len(pix_ids)):
    bx, by, bz = boundaries[i]
    b_lon = np.degrees(np.arctan2(by, bx))
    b_lat = np.degrees(np.arcsin(bz))
    polygons.append(list(zip(b_lon, b_lat)))

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()}, figsize=(10, 7))

vmin, vmax = np.nanmin(values), np.nanmax(values)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

coll = PolyCollection(polygons, transform=ccrs.PlateCarree(),
                       edgecolors="face", linewidths=0.1)
coll.set_array(values)
coll.set_cmap(plt.cm.RdYlBu_r)
coll.set_norm(norm)
ax.add_collection(coll)

# Overlay country outline from earthkit-geo shapes
for shape in shapes:
    s_lons = [pt[1] for pt in shape]
    s_lats = [pt[0] for pt in shape]
    ax.plot(s_lons, s_lats, color="black", linewidth=1.2, transform=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle="--")
ax.set_extent([5, 16, 46.5, 55.5])
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

plt.colorbar(coll, ax=ax, label="K", shrink=0.8, pad=0.02)
ax.set_title(f"polygon ({COUNTRY}) — 2t — 1990-02-01 12:00\n(native HEALPix pixels, no interpolation)")
plt.tight_layout()
plt.show()

In [ ]:
store.clear_cache()